In [124]:
# LIBRERIAS PARA MANEJAR EL DATAFRAME
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# LIBRERIAS PARA LOS MODELOS
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

# LIBRERIAS PARA LAS METRICAS
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# LIMPIEZA Y MANEJO DE DATOS  
import re
import unicodedata

# CONTROL DE ADVERTENCIAS
import warnings
warnings.filterwarnings("ignore")

In [125]:
#BUSCAR EL ARCHIVO A MANEJAR
file_path = 'SIN TOCAR - transactions_full_with_dirty_records.csv'


# CARGAR EL ARCHIVO AL DATAFRAME
df = pd.read_csv(file_path)

# Visualizacion de las dimensiones iniciales
print(df.shape)

# Visualizacion de las primeras filas
print(df.head())

(600000, 31)
  transaction_id customer_id transaction_datetime transaction_date  \
0    TX000442322   CUST04737  2025-01-16 06:18:07       2025-01-16   
1    TX000426517   CUST05210  2025-03-14 20:02:51       2025-03-14   
2    TX000215101   CUST01678  2025-01-06 13:49:10       2025-01-06   
3    TX000478746   CUST01387  2025-01-11 21:31:46       2025-01-11   
4    TX000489506   CUST04801  2025-01-27 13:34:17       2025-01-27   

  transaction_time                   product_type transaction_amount  \
0         06:18:07                         ahorro             3258.3   
1         20:02:51                pago de tarjeta            2324.31   
2         13:49:10                         ahorro            3905.44   
3         21:31:46  transferencia cuenta a cuenta            1272.11   
4         13:34:17  transferencia cuenta a cuenta             3073.1   

  transaction_direction   channel transaction_status  ...  \
0                salida  sucursal         completada  ...   
1          

In [126]:
# VISUALIZACION DE CANTIDAD DE FILAS Y COLUMNAS EN EL DF
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 600000
Columnas: 31


In [127]:
# VISUALIZACION DE LOS NOMBRES DE LAS COLUMNAS DEL DF
print("\nCOLUMNAS:")
print(df.columns.tolist())



COLUMNAS:
['transaction_id', 'customer_id', 'transaction_datetime', 'transaction_date', 'transaction_time', 'product_type', 'transaction_amount', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'monthly_income', 'income_frequency', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel', 'deviation_from_usual_amount_pct', 'anomaly_flag', 'risk_profile', 'risk_score', 'fraud_label', 'data_quality_flag']


In [128]:
#VISUALIZACION DE LAS COLUMNAS Y LOS TIPOS DE DATOS DE CADA UNA DE ELLAS
print("\nTIPOS DE DATOS:")
print(df.dtypes)


TIPOS DE DATOS:
transaction_id                      object
customer_id                         object
transaction_datetime                object
transaction_date                    object
transaction_time                    object
product_type                        object
transaction_amount                  object
transaction_direction               object
channel                             object
transaction_status                  object
merchant_or_destination             object
source_account_type                 object
destination_account_type            object
customer_segment                    object
occupation                          object
industry                            object
employment_type                     object
monthly_income                     float64
income_frequency                    object
customer_tenure_months               int64
avg_3m_transaction_amount          float64
avg_3m_transaction_count             int64
usual_transaction_hour_range        o

In [129]:
# VISUALIZACION DE CATINDADES DE LOS VALORES NULOS EN LAS COLUMNAS
print("\nNULOS POR COLUMNA:")
print(df.isnull().sum())


NULOS POR COLUMNA:
transaction_id                      1729
customer_id                        15594
transaction_datetime               13654
transaction_date                   13477
transaction_time                       0
product_type                       13996
transaction_amount                 13829
transaction_direction                  0
channel                             2094
transaction_status                  2503
merchant_or_destination                0
source_account_type                    0
destination_account_type               0
customer_segment                       0
occupation                         13814
industry                               0
employment_type                        0
monthly_income                     13737
income_frequency                       0
customer_tenure_months                 0
avg_3m_transaction_amount              0
avg_3m_transaction_count               0
usual_transaction_hour_range           0
usual_product_type                   

In [130]:
# AGRUPACION DE LOS CAMPOS COMO CAMPOS LLAVES PARA DETERMINAR DUBLICADOS
subset_cols = [
    "customer_id",
    "transaction_datetime",
    "transaction_amount",
    "product_type",
    "channel"
]


# REALIZACION DE LA VERIFICACION DE LOS NULOS TOMANDO ENCUENTA LOS CAMPOS LLAVES
dup_suspect = df[df.duplicated(subset=subset_cols, keep=False)]

# VISUALIZACION DE LOS REGISTROS DUPLICADOS Y POSIBLES SUSPECHOSOS, CON VERIFICACION DE LOS DUPLICACOS POR CATEGORIA COMO SE MUESTRA EN LOS PRINT
print("Duplicados sospechosos (mismo evento sin ID):", dup_suspect.shape[0])
print("Total filas:", len(df))
print("Duplicados exactos:", df.duplicated().sum())
print("Duplicados transaction_id:", df.duplicated(subset=['transaction_id']).sum())
print("Duplicados sospechosos:", df.duplicated(subset=subset_cols).sum())

Duplicados sospechosos (mismo evento sin ID): 76746
Total filas: 600000
Duplicados exactos: 4309
Duplicados transaction_id: 99997
Duplicados sospechosos: 43194


#### 1. Proceso de duplicados (Arlene)

#### Proceso de duplicados (Arlene)

Debido a la naturaleza financiera del dataset, no se eliminó ningún registro duplicado. En su lugar, se identificaron y marcaron posibles inconsistencias. Esta estrategia preserva la trazabilidad histórica de las transacciones y evita la pérdida de información potencialmente relevante para auditoría y detección de fraude.

**Detección de duplicados:**
- Se realizó una identificación de registros duplicados utilizando todas las columnas relevantes.

**Decisión de negocio (NO eliminación)**
- Debido a la naturaleza financiera del dataset, no se eliminó ningún registro duplicado.
- Se evita la pérdida de información crítica para auditoría, trazabilidad y análisis de fraude.

El dataset conserva todas las filas originales y columnas originales.

In [131]:
# OTRA LIBRERIA PARA UTILIZAR PARA LA LIMPIEZA DE DATOS Y HACER ALGUNOS RESETS
# import unicodedata #Adison

print("HOMOGENIZANDO COLUMNAS DE TEXTO")

text_columns = [
    "product_type",
    "transaction_direction",
    "channel",
    "transaction_status",
    "customer_segment",
    "occupation",
    "industry",
    "employment_type",
    "income_frequency",
    "usual_product_type",
    "usual_channel",
    "risk_profile",
    "data_quality_flag"
]


HOMOGENIZANDO COLUMNAS DE TEXTO


In [132]:

# Función auxiliar para remover acentos de forma segura, PARA OBTENER UNA HOMOGENIZACION DE LOS DATOS
def remover_acentos(texto):
    if pd.isna(texto) or texto == 'nan':
        return texto
    # Normaliza el texto para separar las letras de sus acentos y filtra los caracteres de acentuación
    return "".join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

# RECORREMOS EL DF COMPLETO UTILIZACON LA FUNCION Y LOGRAR LA HOMOGENIZACION
for col in text_columns:
    if col in df.columns:
        #  Convertir a string, remover espacios en blanco extremos y pasar a minúsculas
        df[col] = df[col].astype(str).str.strip().str.lower()
        
        #  Remover acentos y tildes para estandarizar la codificación
        df[col] = df[col].apply(remover_acentos)
        
        #  Corregir efectos secundarios de la conversión a string en datos faltantes
        # Si quedó como cadena vacía "" o como "nan", lo estandarizamos a "unknown"
        df[col] = df[col].replace(["", "nan"], "unknown")

# FIN DEL PROCESO DE Homogeneización DE LOS DATOS
print("¡Homogeneización de texto completada con éxito!")

¡Homogeneización de texto completada con éxito!


#### Proceso de Homogeneización de Texto (Adison)

La homogeneización de texto se realizó con el objetivo de estandarizar las variables categóricas y reducir inconsistencias semánticas dentro del dataset financiero. Para ello, se aplicaron técnicas de limpieza utilizando:

.str.strip() para eliminar espacios en blanco al inicio y final de los textos.
.str.lower() para convertir todos los valores a minúsculas.
Eliminación de acentos y caracteres especiales.
Reemplazo de cadenas vacías, valores nulos y textos como "nan" por "unknown".

Este proceso fue necesario porque en bases de datos financieras integradas desde múltiples fuentes es común encontrar variaciones de escritura para una misma categoría, por ejemplo:

- "Crédito"
- "credito"
- "credito "
- "CREDITO"

Aunque representan el mismo concepto, una computadora los interpreta como categorías diferentes, generando fragmentación y ruido en los datos.

La estandarización textual permitió:

- Reducir la creación de categorías duplicadas o falsas.
- Mejorar la consistencia semántica del dataset.
- Facilitar los procesos posteriores de codificación (encoding).
- Evitar la generación innecesaria de columnas redundantes.
- Mejorar la calidad analítica y el desempeño de modelos de Machine Learning.

Además, esta limpieza ayuda a que los algoritmos identifiquen patrones reales de comportamiento financiero y reduzcan errores derivados de inconsistencias textuales.

In [133]:
# PROCESAREMOS LOS CAMPOS FECHA PARA LOGRAR OBTENER 
# UN SOLO FORMATO AL IGUAL QUE LA VERIFICACION DE QUE NO ESTEN NULOS

print("CONVIRTIENDO FECHAS")

# AGRUPACION DE LOS CAMPOS DE FECHAS DENTRO DEL DF
date_columns = [
    "transaction_datetime",
    "transaction_date"
]


# RECORRER EL DF COMPLETO PARA ESTANDARIZAR LOS CAMPOS FECHA
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

#antes_fecha = df.shape[0]

# eliminar fechas inválidas
#df = df.dropna(subset=["transaction_datetime"])

#despues_fecha = df.shape[0]

#print(f"Filas eliminadas por fechas inválidas: {antes_fecha - despues_fecha}")


CONVIRTIENDO FECHAS


## Fechas Invalidas:
### Consideramos importante mantener las transacciones con fechas "invalidas", porque representan aun asi transacciones existentes. 
### Fechas duplicadas podrian representar, movimientos adicionales reales.
### Desicion: En base al criterio y naturaleza del dataset, preservaremos las transacciones, y realizar en su lugar una estandarizacion de las fechas y horas para cada transaccion.

In [134]:
# MANEJO DE VARIABLES PARA CONVERTIR A DATETIME
# VERIFICACION DE FECHAS INVALIDAS EN transaction_datetime

df["transaction_datetime"] = pd.to_datetime(df['transaction_datetime'],errors="coerce")

fechas_invalidas = df[df['transaction_datetime'].isnull()]
print("Registros con transaction_datetime inválido o nulo:")

print(fechas_invalidas.shape)


Registros con transaction_datetime inválido o nulo:
(17329, 31)


### 2. Proceso de ajuste/corrección de fechas/horas (Daniel)

In [135]:
# VISUALIZACION DE LAS COLUMNAS ACTUALES DE FECHA Y HORA
print("DIAGNÓSTICO — ESTADO ACTUAL DE COLUMNAS DE FECHAS/HORAS")
print(f"Nulos en transaction_datetime : {df['transaction_datetime'].isnull().sum():>10,}")
print(f"Nulos en transaction_date     : {df['transaction_date'].isnull().sum():>10,}")
print(f"Nulos en transaction_time     : {df['transaction_time'].isnull().sum():>10,}")
print(f"\nTipo transaction_datetime : {df['transaction_datetime'].dtype}")
print(f"Tipo transaction_date     : {df['transaction_date'].dtype}")
print(f"Tipo transaction_time     : {df['transaction_time'].dtype}")
print(df[["transaction_time"]].head())



DIAGNÓSTICO — ESTADO ACTUAL DE COLUMNAS DE FECHAS/HORAS
Nulos en transaction_datetime :     17,329
Nulos en transaction_date     :     17,152
Nulos en transaction_time     :          0

Tipo transaction_datetime : datetime64[ns]
Tipo transaction_date     : datetime64[ns]
Tipo transaction_time     : object
  transaction_time
0         06:18:07
1         20:02:51
2         13:49:10
3         21:31:46
4         13:34:17


Este bloque estandariza la columna transaction_time usando expresiones regulares. Convierte horas válidas al formato HH:MM:SS y marca como nulos los valores inválidos. Desde el punto de vista bancario, mejora la calidad de una variable temporal importante para detectar patrones sospechosos. Desde ciencia de datos, prepara la columna para futuras transformaciones y creación de variables derivadas.

In [136]:
# USO DE LA LIBRERIA PARA MANEJO DE TEXTO CON PATRONES
# import re

# FUNCION PARA EL MANEJO DE LA LIBRERIA EN LOS TEXTOS DESEADOS EN BUSQUEDAS Y VALIDACIONES
def estandarizar_hora(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip()
    if re.fullmatch(r'\d{1,2}:\d{2}:\d{2}', s):
        h, m, seg = map(int, s.split(':'))
        if 0 <= h <= 23 and 0 <= m <= 59 and 0 <= seg <= 59:
            return f"{h:02d}:{m:02d}:{seg:02d}"
    if re.fullmatch(r'\d{1,2}:\d{2}', s):
        h, m = map(int, s.split(':'))
        if 0 <= h <= 23 and 0 <= m <= 59:
            return f"{h:02d}:{m:02d}:00"
    return np.nan

# COMPARACION DE INVALIDOS ANTES Y DESPUES DE LA TRANSFORMACION
invalidos_antes = df['transaction_time'].isnull().sum()
df['transaction_time'] = df['transaction_time'].apply(estandarizar_hora)
invalidos_despues = df['transaction_time'].isnull().sum()

print(f"transaction_time inválidos antes : {invalidos_antes}")
print(f"transaction_time inválidos después: {invalidos_despues}")
print(f"Formatos inválidos detectados    : {invalidos_despues - invalidos_antes}")


transaction_time inválidos antes : 0
transaction_time inválidos después: 3816
Formatos inválidos detectados    : 3816


In [137]:
# Asegurar que transaction_date esté en formato datetime
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")

# Asegurar que transaction_time esté en formato texto
df["transaction_time"] = df["transaction_time"].fillna("00:00:00").astype(str)

nulos_antes = df['transaction_datetime'].isnull().sum()

mask_recover = df['transaction_datetime'].isnull() & df['transaction_date'].notnull()
hora_str  = df.loc[mask_recover, 'transaction_time'].fillna('00:00:00')
fecha_str = df.loc[mask_recover, 'transaction_date'].dt.strftime('%Y-%m-%d')

df.loc[mask_recover, 'transaction_datetime'] = pd.to_datetime(fecha_str + ' ' + hora_str, errors='coerce')

nulos_despues = df['transaction_datetime'].isnull().sum()
print(f"Nulos en transaction_datetime antes   : {nulos_antes:,}")
print(f"Registros recuperados                 : {nulos_antes - nulos_despues:,}")
print(f"Nulos en transaction_datetime después : {nulos_despues:,}")

Nulos en transaction_datetime antes   : 17,329
Registros recuperados                 : 11,058
Nulos en transaction_datetime después : 6,271


In [138]:

# Asegurar que transaction_datetime esté en formato datetime
df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"], errors="coerce")
nulos_antes_date = df['transaction_date'].isnull().sum()

mask_date = df['transaction_date'].isnull() & df['transaction_datetime'].notnull()
df.loc[mask_date, 'transaction_date'] = df.loc[mask_date, 'transaction_datetime'].dt.normalize()

nulos_despues_date = df['transaction_date'].isnull().sum()
print(f"Nulos en transaction_date antes  : {nulos_antes_date:,}")
print(f"Registros recuperados            : {nulos_antes_date - nulos_despues_date:,}")
print(f"Nulos en transaction_date después: {nulos_despues_date:,}")

Nulos en transaction_date antes  : 17,152
Registros recuperados            : 10,881
Nulos en transaction_date después: 6,271


In [139]:
FECHA_INICIO = pd.Timestamp('2025-01-01')
FECHA_FIN    = pd.Timestamp('2025-03-31 23:59:59')

mask_valido  = df['transaction_datetime'].notna()
mask_antes   = mask_valido & (df['transaction_datetime'] < FECHA_INICIO)
mask_despues = mask_valido & (df['transaction_datetime'] > FECHA_FIN)
mask_fuera   = mask_antes | mask_despues

mask_actualizar = mask_fuera & (df['data_quality_flag'] == 'valid')

print(f"Fechas anteriores al 2025-01-01 : {mask_antes.sum():,}")
print(f"Fechas posteriores al 2025-03-31: {mask_despues.sum():,}")
print(f"Total fuera del rango valido    : {mask_fuera.sum():,}")
print(f"Registros actualizados en flag  : {mask_actualizar.sum():,}")

df.loc[mask_actualizar, 'data_quality_flag'] = 'invalid_format'

print("\nSolo se actualizo data_quality_flag en registros que eran 'valid'.")
print("Nota: no se eliminaron filas.")

Fechas anteriores al 2025-01-01 : 0
Fechas posteriores al 2025-03-31: 0
Total fuera del rango valido    : 0
Registros actualizados en flag  : 0

Solo se actualizo data_quality_flag en registros que eran 'valid'.
Nota: no se eliminaron filas.


In [140]:
mask_dt_valido = df['transaction_datetime'].notna()
df.loc[mask_dt_valido, 'transaction_date'] = df.loc[mask_dt_valido, 'transaction_datetime'].dt.normalize()

mask_time_nulo = mask_dt_valido & df['transaction_time'].isnull()
df.loc[mask_time_nulo, 'transaction_time'] = df.loc[mask_time_nulo, 'transaction_datetime'].dt.strftime('%H:%M:%S')
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S', errors='coerce').dt.time

print("=" * 55)
print("RESUMEN FINAL — COLUMNAS DE FECHAS/HORAS")
print("=" * 55)
print(f"Nulos en transaction_datetime : {df['transaction_datetime'].isnull().sum():,}")
print(f"Nulos en transaction_date     : {df['transaction_date'].isnull().sum():,}")
print(f"Nulos en transaction_time     : {df['transaction_time'].isnull().sum():,}")
print(f"\nFecha mínima registrada: {df['transaction_datetime'].min()}")
print(f"Fecha máxima registrada: {df['transaction_datetime'].max()}")
print(f"Total de filas en el dataset : {df.shape[0]:,}")
print(df[["transaction_time"]].head())
print(df['transaction_time'].dropna().head(3).tolist())

RESUMEN FINAL — COLUMNAS DE FECHAS/HORAS
Nulos en transaction_datetime : 6,271
Nulos en transaction_date     : 6,271
Nulos en transaction_time     : 0

Fecha mínima registrada: 2025-01-01 06:00:00
Fecha máxima registrada: 2025-03-31 22:59:57
Total de filas en el dataset : 600,000
  transaction_time
0         06:18:07
1         20:02:51
2         13:49:10
3         21:31:46
4         13:34:17
[datetime.time(6, 18, 7), datetime.time(20, 2, 51), datetime.time(13, 49, 10)]


#### Proceso de ajuste/corrección de fechas/horas (Daniel)

El proceso de ajuste de fechas y horas se realizó con el objetivo de estandarizar y recuperar los datos temporales del dataset financiero sin eliminar registros. Dado que cada fila representa una transacción con valor económico, se priorizó la recuperación sobre el descarte.

**Cómo se hizo:**
Se aplicaron cuatro pasos secuenciales. Primero, se validó y estandarizó `transaction_time` al formato `HH:MM:SS`, marcando como `NaN` los valores con horas, minutos o segundos fuera de rango (ej. `25:00:00`). Segundo, se reconstruyó `transaction_datetime` combinando `transaction_date` + `transaction_time` en los registros donde el datetime era nulo pero las columnas individuales estaban disponibles. Tercero, se derivó `transaction_date` desde `transaction_datetime` para los registros con fecha nula. Finalmente, se validó el rango temporal permitido (2025-01-01 a 2025-03-31) actualizando `data_quality_flag` a `invalid_format` únicamente en registros que tenían el flag original en `valid`, preservando sin modificar los flags preexistentes como `dirty`, `duplicate` o `null_issue`. En este dataset el resultado fue 0 registros fuera de rango, confirmando que todas las fechas válidas están dentro del período esperado.

**Por qué se hizo:**
Las columnas `transaction_datetime`, `transaction_date` y `transaction_time` son críticas para análisis de comportamiento del cliente y detección de fraude. Un registro sin fecha válida pierde trazabilidad, pero uno con fecha reconstruida sigue aportando información útil. En datasets financieros, eliminar registros implica pérdida de información sobre movimientos reales de dinero.

**Para qué sirve:**
Garantizar la integridad temporal permite que los modelos de Machine Learning identifiquen correctamente patrones de uso por hora, día y mes. Una fecha correcta es esencial para validar si una transacción ocurrió dentro del comportamiento habitual del cliente y para calcular métricas de detección de anomalías y fraude con mayor precisión.

In [141]:
print("--- VISUALIZACIÓN DE VALORES NULOS (Adison) ---")

# 1. Total de nulos por columna y su porcentaje
df_nulos = pd.DataFrame({
    'Nulos Totales': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100) # Muestra la proporción real
})

# Filtrar para mostrar solo las columnas que sí tienen nulos
df_nulos = df_nulos[df_nulos['Nulos Totales'] > 0]

print("Columnas con datos faltantes:")
print(df_nulos)

--- VISUALIZACIÓN DE VALORES NULOS (Adison) ---
Columnas con datos faltantes:
                      Nulos Totales  Porcentaje (%)
transaction_id                 1729        0.288167
customer_id                   15594        2.599000
transaction_datetime           6271        1.045167
transaction_date               6271        1.045167
transaction_amount            13829        2.304833
monthly_income                13737        2.289500


#### Proceso de manejo de datos nulos (Adison)

En este paso, se usó el código .isnull().sum() únicamente para visualizar la cantidad de vacíos en cada columna sin alterar el dataset. Se decidió *no borrar ninguna fila* porque eliminar masivamente los registros con faltantes destruiría miles de transacciones importantes de las 600,000 originales, lo que dejaría sin datos suficientes a los modelos de Machine Learning para entrenarse. El único objetivo fue diagnosticar el estado actual de la base de datos de forma segura, cuidando la información financiera antes de decidir cómo rellenarla.

In [142]:
# ============================================================
# 5. PROCESO DE MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS (RONI)
# ============================================================

print("\n" + "=" * 70)
print("5. MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS - RONI")
print("=" * 70)

# Validación de seguridad: asegura que el DataFrame exista antes de continuar.
if "df" not in globals():
    raise NameError("El DataFrame 'df' no está definido. Ejecuta primero la celda de carga del dataset.")

print(f"Dimensiones actuales del dataset: {df.shape}")

# 1. Columnas numéricas esperadas según la estructura del dataset financiero.
columnas_numericas_esperadas = [
    "transaction_amount",
    "monthly_income",
    "customer_tenure_months",
    "avg_3m_transaction_amount",
    "avg_3m_transaction_count",
    "deviation_from_usual_amount_pct",
    "risk_score",
    "anomaly_flag",
    "fraud_label"
]

# 2. Seleccionar solamente las columnas que realmente existen en el dataset.
columnas_numericas = [col for col in columnas_numericas_esperadas if col in df.columns]
columnas_no_encontradas = [col for col in columnas_numericas_esperadas if col not in df.columns]

print("\nColumnas numéricas encontradas:")
print(columnas_numericas)

if columnas_no_encontradas:
    print("\nColumnas esperadas que no fueron encontradas en el dataset:")
    print(columnas_no_encontradas)

# Si ninguna columna esperada existe, se detiene el proceso con un mensaje claro.
if len(columnas_numericas) == 0:
    raise ValueError("No se encontraron columnas numéricas esperadas. Revisa los nombres de columnas del dataset.")

# 3. Diagnóstico inicial antes del ajuste.
print("\nTipos de datos antes del ajuste:")
print(df[columnas_numericas].dtypes)

print("\nNulos antes del ajuste:")
print(df[columnas_numericas].isnull().sum())

# 4. Conversión segura a formato numérico.
# errors='coerce' convierte valores no numéricos en NaN para poder tratarlos.
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nTipos de datos después de convertir a numérico:")
print(df[columnas_numericas].dtypes)

# 5. Imputación de nulos numéricos con la mediana.
# La mediana es más robusta ante valores extremos que la media.
print("\nImputación de valores nulos con la mediana:")
for col in columnas_numericas:
    nulos = df[col].isnull().sum()
    if nulos > 0:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)
        print(f"{col}: {nulos} nulos reemplazados con mediana = {mediana}")
    else:
        print(f"{col}: sin nulos")

# 6. Validación de valores negativos donde no deberían existir.
columnas_sin_negativos = [
    "transaction_amount",
    "monthly_income",
    "customer_tenure_months",
    "avg_3m_transaction_amount",
    "avg_3m_transaction_count",
    "risk_score"
]

print("\nValidación de valores negativos:")
for col in columnas_sin_negativos:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        print(f"{col}: {negativos} valores negativos encontrados")

        if negativos > 0:
            mediana = df.loc[df[col] >= 0, col].median()
            df.loc[df[col] < 0, col] = mediana
            print(f"{col}: negativos reemplazados por la mediana de valores válidos = {mediana}")

# 7. Validación de columnas binarias.
columnas_binarias = ["anomaly_flag", "fraud_label"]

print("\nValidación de columnas binarias:")
for col in columnas_binarias:
    if col in df.columns:
        valores_unicos = sorted(df[col].dropna().unique())
        print(f"{col} - valores únicos antes de validar: {valores_unicos}")

        valores_invalidos = ~df[col].isin([0, 1])
        cantidad_invalidos = valores_invalidos.sum()

        if cantidad_invalidos > 0:
            moda = df.loc[df[col].isin([0, 1]), col].mode()
            moda = moda.iloc[0] if len(moda) > 0 else 0
            df.loc[valores_invalidos, col] = moda
            print(f"{col}: {cantidad_invalidos} valores inválidos reemplazados con la moda = {moda}")

# 8. Validación del rango lógico de risk_score.
if "risk_score" in df.columns:
    fuera_rango = ((df["risk_score"] < 0) | (df["risk_score"] > 100)).sum()
    print(f"\nrisk_score fuera del rango 0-100: {fuera_rango}")

    if fuera_rango > 0:
        df["risk_score"] = df["risk_score"].clip(lower=0, upper=100)
        print("risk_score ajustado al rango permitido entre 0 y 100.")

# 9. Resumen final.
print("\nNulos finales en columnas numéricas:")
print(df[columnas_numericas].isnull().sum())

print("\nResumen estadístico final de columnas numéricas:")
display(df[columnas_numericas].describe().T)

print("\nProceso de manejo/ajuste de columnas numéricas completado correctamente.")



5. MANEJO / AJUSTE DE COLUMNAS NUMÉRICAS - RONI
Dimensiones actuales del dataset: (600000, 31)

Columnas numéricas encontradas:
['transaction_amount', 'monthly_income', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'deviation_from_usual_amount_pct', 'risk_score', 'anomaly_flag', 'fraud_label']

Tipos de datos antes del ajuste:
transaction_amount                  object
monthly_income                     float64
customer_tenure_months               int64
avg_3m_transaction_amount          float64
avg_3m_transaction_count             int64
deviation_from_usual_amount_pct    float64
risk_score                           int64
anomaly_flag                         int64
fraud_label                          int64
dtype: object

Nulos antes del ajuste:
transaction_amount                 13829
monthly_income                     13737
customer_tenure_months                 0
avg_3m_transaction_amount              0
avg_3m_transaction_count               0
de

,count,mean,std,min,25%,50%,75%,max
transaction_amount,600000.0,33347.127840,276848.637351,50.00,1941.5275,3566.13,7023.0025,4999709.93
monthly_income,600000.0,61763.816113,45279.870562,7.08,30830.9300,49434.92,82817.5400,219782.40
customer_tenure_months,600000.0,61.991647,34.112622,3.00,32.0000,62.00,92.0000,120.00
avg_3m_transaction_amount,600000.0,4271.425676,3441.351155,245.14,1909.8700,3265.99,5652.3300,23041.93
avg_3m_transaction_count,600000.0,92.518910,83.736829,0.00,66.0000,76.00,128.0000,1500.00
deviation_from_usual_amount_pct,600000.0,212.712029,722.314789,-99.75,-20.6900,4.93,33.9100,6856.13
risk_score,600000.0,100.000000,0.000000,100.00,100.0000,100.00,100.0000,100.00
anomaly_flag,600000.0,0.129835,0.336122,0.00,0.0000,0.00,0.0000,1.00
fraud_label,600000.0,0.099788,0.299718,0.00,0.0000,0.00,0.0000,1.00



Proceso de manejo/ajuste de columnas numéricas completado correctamente.


#### 5. Proceso de manejo/ajuste de columnas numéricas (Roni)

El proceso de manejo y ajuste de columnas numéricas se realizó con el objetivo de asegurar que las variables cuantitativas del dataset financiero estuvieran correctamente formateadas, limpias y listas para las siguientes etapas del proyecto.

**Cómo se hizo:**  
Primero se identificaron las columnas numéricas esperadas dentro del dataset, tales como `transaction_amount`, `monthly_income`, `customer_tenure_months`, `avg_3m_transaction_amount`, `avg_3m_transaction_count`, `deviation_from_usual_amount_pct`, `risk_score`, `anomaly_flag` y `fraud_label`. Luego se validó cuáles de estas columnas existían realmente en el DataFrame para evitar errores por nombres no encontrados. Posteriormente, las columnas fueron convertidas a formato numérico utilizando `pd.to_numeric()` con `errors="coerce"`, lo que permitió transformar valores inválidos en valores nulos controlados. Después se imputaron los valores faltantes utilizando la mediana, se revisaron valores negativos en columnas donde no tienen sentido lógico y se validaron las columnas binarias para asegurar que solo contuvieran valores 0 y 1.

**Por qué se hizo:**  
Este ajuste fue necesario porque las columnas numéricas son fundamentales para el análisis financiero, la detección de anomalías y la construcción de modelos de Machine Learning. Si estas variables contienen textos, valores inválidos, nulos sin tratar, negativos incorrectos o formatos inconsistentes, pueden generar errores en los cálculos, afectar los análisis estadísticos y reducir la calidad de los modelos predictivos. Además, en un dataset financiero, variables como el monto de la transacción, el ingreso mensual, la antigüedad del cliente y el puntaje de riesgo deben mantener coherencia lógica.

**Para qué se hizo:**  
Este proceso permite dejar el dataset en mejores condiciones para etapas posteriores como la detección de outliers, el análisis exploratorio, la generación de reportes y el modelado predictivo de fraude. Al estandarizar y validar las columnas numéricas, se mejora la confiabilidad de los datos, se reducen errores de procesamiento y se garantiza que las variables utilizadas representen correctamente el comportamiento financiero de las transacciones.


#### Detección de outliers (Felix)

Realizamos un proceso de detección de outliers en los montos de transacciones, utilizando un método estadístico para identificar registros fuera del comportamiento normal.

Este análisis se llevó a cabo con el objetivo de mejorar la calidad de los datos, detectar posibles errores o anomalías y garantizar resultados más confiables en los reportes y análisis posteriores.

In [143]:

# DETECCION DE LOS OUTLIERS
print("\n" + "=" * 50)
print("DETECTANDO OUTLIERS")
print("=" * 50)

df["transaction_datetime"] = pd.to_datetime(df['transaction_datetime'],errors="coerce")

Q1 = df["transaction_amount"].quantile(0.25)
Q3 = df["transaction_amount"].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[
    (df["transaction_amount"] < limite_inferior) |
    (df["transaction_amount"] > limite_superior)
]

print(f"Cantidad de outliers detectados: {outliers.shape[0]}")


DETECTANDO OUTLIERS
Cantidad de outliers detectados: 66425



### 1. Proceso de los duplicados. (Arlene)
### 2. Proceso de ajuste/correcion de fechas/horas (Daniel)
### 3. Proceso de manejo de datos nulos (Adison)
### 4. Proceso de homogenizar de texto (Adison)
### 5. Proceso de manejo/ajuste de columnas numericas (Roni)
### 6. Deteccion de outliers (Felix)
### 7. Repo de Github (Daniel)
### Disclaimer de la bitacora de CADA PUNTO, haganlo como un registro detallado. 3 puntos a responder: como, por que y para que

In [144]:
# GRUPO 2 

#Este Jupiter incluye:

# - Carga y limpieza de datos
# - Preparación para Machine Learning
# - Modelo Random Forest
# - Modelo Logistic Regression
# - Modelo K-Means
# - Evaluación de modelos
# - Visualización de resultados

# - **Variable objetivo elegida:** `fraud_label` — etiqueta binaria (0 = regular, 1 = fraudulento)
# - **Estrategia:** Comparar 5 modelos distintos y seleccionar el de mejor ROC AUC
# - **Modelos evaluados:** Random Forest, Logistic Regression, K-Means, XGBoost, Isolation Forest


In [145]:
#Bloque de importación de librerías
#En este bloque se cargan las librerías necesarias para trabajar el proyecto de detección de fraude financiero.
#Se utilizan herramientas de manipulación de datos, preparación de variables, división de entrenamiento y prueba,
#entrenamiento de modelos y evaluación de resultados.
#¿Por qué se usa este bloque?
#Porque antes de entrenar cualquier modelo es necesario tener disponibles las funciones que permiten:
#• leer y manipular el dataset;
#• preparar las variables predictoras;
#• dividir los datos en entrenamiento y prueba;
#• entrenar modelos supervisados y no supervisados;
#• medir el rendimiento de cada modelo.
#Este bloque es la base técnica del proyecto.



In [146]:
# ==========================================
# VARIABLE OBJETIVO
# ==========================================

# Este bloque define fraud_label como variable objetivo del proyecto. 
# Aunque el proyecto iniciará con modelos no supervisados,
# fraud_label se conserva como referencia para evaluar posteriormente si las anomalías detectadas coinciden con fraudes reales. 
# En la etapa supervisada, esta variable será utilizada directamente para entrenar los modelos de clasificación.
# Esta variable representa si una transacción fue fraudulenta o no:
# 0 = transacción regular
# 1 = transacción fraudulenta

target = "fraud_label"

# Validación de seguridad
if target not in df.columns:
    raise KeyError(f"La variable objetivo '{target}' no existe en el DataFrame.")

print(f"Variable objetivo definida correctamente: {target}")
print("Distribución de la variable objetivo:")
print(df[target].value_counts(dropna=False))

Variable objetivo definida correctamente: fraud_label
Distribución de la variable objetivo:
fraud_label
0    540127
1     59873
Name: count, dtype: int64


In [147]:
#Bloque de filtrado de datos válidos
# Este bloque filtra los registros válidos según la columna data_quality_flag. 
# Los registros con valor "valid" se conservan en df_model para ser usados en el proceso de modelado,
# mientras que los registros no válidos se separan en df_invalidos para fines de auditoría o revisión posterior.
# Desde el punto de vista bancario, esta separación permite entrenar modelos con 
# información más confiable sin perder trazabilidad de los registros excluidos. 

# ==========================================
# FILTRAR SOLO DATOS VÁLIDOS
# ==========================================

# Validación de seguridad
if "data_quality_flag" not in df.columns:
    raise KeyError("La columna 'data_quality_flag' no existe en el DataFrame.")

# Asegurar formato homogéneo antes del filtro
df["data_quality_flag"] = (
    df["data_quality_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Distribución de data_quality_flag antes del filtro:")
print(df["data_quality_flag"].value_counts(dropna=False))

# Separar registros válidos para modelado
df_model = df[
    df["data_quality_flag"] == "valid"
].copy()

# Conservar registros no válidos para auditoría
df_invalidos = df[
    df["data_quality_flag"] != "valid"
].copy()

print("\nDimensiones del dataset original:")
print(df.shape)

print("\nDimensiones del dataset válido para modelado:")
print(df_model.shape)

print("\nRegistros excluidos del modelado:")
print(df_invalidos.shape)

Distribución de data_quality_flag antes del filtro:
data_quality_flag
valid             500000
null_issue         37554
duplicate          22495
dirty              22491
invalid_format     17460
Name: count, dtype: int64

Dimensiones del dataset original:
(600000, 31)

Dimensiones del dataset válido para modelado:
(500000, 31)

Registros excluidos del modelado:
(100000, 31)


In [148]:
# Este bloque identifica las columnas que no deben utilizarse como variables predictoras. 
# Se excluyen identificadores únicos, fechas crudas, variables de calidad de datos, 
# la variable objetivo y columnas que podrían representar fuga de información. 
# Debido a que el flujo del proyecto se reorganiza para iniciar con modelos no supervisados y luego modelos supervisados, 
# se recomienda separar las columnas excluidas para cada enfoque.
# Esto mejora la trazabilidad metodológica y evita que variables sensibles o derivadas contaminen el entrenamiento de los modelos.
# Columnas excluidas para modelos no supervisados
# Estos modelos no deben usar fraud_label porque no aprenden con etiqueta.
columnas_no_predictoras_no_supervisado = [
    "fraud_label",
    "anomaly_flag",
    "risk_score",
    "risk_profile",
    "deviation_from_usual_amount_pct",
    "data_quality_flag",
    "transaction_id",
    "customer_id",
    "transaction_datetime",
    "transaction_date",
    "transaction_time"
]

# Columnas excluidas para modelos supervisados
# fraud_label se separa como variable objetivo y no debe entrar dentro de X.
columnas_no_predictoras_supervisado = [
    "fraud_label",
    "anomaly_flag",
    "risk_score",
    "risk_profile",
    "deviation_from_usual_amount_pct",
    "data_quality_flag",
    "transaction_id",
    "customer_id",
    "transaction_datetime",
    "transaction_date",
    "transaction_time"
]

print("Columnas no predictoras para modelos no supervisados:")
print(columnas_no_predictoras_no_supervisado)

print("\nColumnas no predictoras para modelos supervisados:")
print(columnas_no_predictoras_supervisado)

Columnas no predictoras para modelos no supervisados:
['fraud_label', 'anomaly_flag', 'risk_score', 'risk_profile', 'deviation_from_usual_amount_pct', 'data_quality_flag', 'transaction_id', 'customer_id', 'transaction_datetime', 'transaction_date', 'transaction_time']

Columnas no predictoras para modelos supervisados:
['fraud_label', 'anomaly_flag', 'risk_score', 'risk_profile', 'deviation_from_usual_amount_pct', 'data_quality_flag', 'transaction_id', 'customer_id', 'transaction_datetime', 'transaction_date', 'transaction_time']


In [149]:


# Este bloque es necesario porque define formalmente qué variables entrarán a los modelos.
# La separación entre columnas predictoras no supervisadas
# y supervisadas fortalece la metodología del proyecto y
# permite mantener el orden correcto: primero modelos no supervisados y luego modelos supervisados.

# COLUMNAS PREDICTORAS
# ==========================================

# Columnas que se usarán en modelos no supervisados
columnas_predictoras_no_supervisado = [
    col for col in df_model.columns
    if col not in columnas_no_predictoras_no_supervisado
]

# Columnas que se usarán en modelos supervisados
columnas_predictoras_supervisado = [
    col for col in df_model.columns
    if col not in columnas_no_predictoras_supervisado
]

print("Columnas predictoras para modelos no supervisados:")
print(columnas_predictoras_no_supervisado)

print("\nTotal columnas predictoras no supervisadas:")
print(len(columnas_predictoras_no_supervisado))

print("\nColumnas predictoras para modelos supervisados:")
print(columnas_predictoras_supervisado)

print("\nTotal columnas predictoras supervisadas:")
print(len(columnas_predictoras_supervisado))

Columnas predictoras para modelos no supervisados:
['product_type', 'transaction_amount', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'monthly_income', 'income_frequency', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel']

Total columnas predictoras no supervisadas:
20

Columnas predictoras para modelos supervisados:
['product_type', 'transaction_amount', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'monthly_income', 'income_frequency', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'usual_transaction_hour_range', 'usual_product_type', 'usua

In [150]:
#Bloque de revisión de registros problemáticos
#En este bloque se visualizan los registros que no cumplen con la calidad esperada.
#¿Por qué se usa este bloque?
#Porque permite revisar los datos problemáticos sin eliminarlos.
#Esto es importante en proyectos financieros, ya que los registros con errores pueden contener información útil para auditoría, 
# revisión operativa o mejora del proceso de captura de datos.
#Este bloque no busca entrenar el modelo, sino documentar y entender la calidad del dataset.

#VER REGISTROS PROBLEMATICOS SIN ELIMINARLOS
invalidos = df[df["data_quality_flag"] != "valid"]

invalidos.head()

print("Cantidad de registros problemáticos:", invalidos.shape[0])
print(invalidos["data_quality_flag"].value_counts())

Cantidad de registros problemáticos: 100000
data_quality_flag
null_issue        37554
duplicate         22495
dirty             22491
invalid_format    17460
Name: count, dtype: int64


In [151]:
# Este bloque crea el conjunto de variables que será utilizado por los modelos no supervisados. 
# Para esto, toma del DataFrame df_model únicamente las columnas definidas como predictoras para el enfoque no supervisado

# DEFINIR X PARA MODELOS NO SUPERVISADOS

# Los modelos no supervisados no utilizan y para entrenar.
# Solo usan variables predictoras sin fraud_label.

X_no_supervisado = df_model[columnas_predictoras_no_supervisado].copy()

print("Dimensiones de X_no_supervisado:")
print(X_no_supervisado.shape)

print("\nColumnas usadas en modelos no supervisados:")
print(X_no_supervisado.columns.tolist())

# Variable auxiliar para evaluación posterior
# No se usará para entrenar Isolation Forest ni K-Means.
y_auxiliar = df_model[target].copy()

print("\nDimensiones de y_auxiliar:")
print(y_auxiliar.shape)

Dimensiones de X_no_supervisado:
(500000, 20)

Columnas usadas en modelos no supervisados:
['product_type', 'transaction_amount', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'monthly_income', 'income_frequency', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel']

Dimensiones de y_auxiliar:
(500000,)


In [152]:
# Este bloque transforma X_no_supervisado en una matriz numérica y escalada lista para ser utilizada
# por modelos no supervisados como Isolation Forest y K-Means.
# PLos modelos no supervisados no pueden trabajar directamente con variables de texto. 
# Por eso las variables categóricas se convierten en variables dummy y las variables numéricas se escalan.

#  PREPARACIÓN DE DATOS PARA MODELOS NO SUPERVISADOS


print("PREPARANDO DATOS PARA MODELOS NO SUPERVISADOS")

# Validación de seguridad
if "X_no_supervisado" not in globals():
    raise NameError("X_no_supervisado no está definido. Ejecuta primero el bloque anterior.")

# Crear copia de trabajo
X_no_supervisado_preparado = X_no_supervisado.copy()

# Identificar columnas categóricas y numéricas
columnas_categoricas_no_sup = X_no_supervisado_preparado.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

columnas_numericas_no_sup = X_no_supervisado_preparado.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("\nColumnas numéricas:")
print(columnas_numericas_no_sup)

print("\nColumnas categóricas:")
print(columnas_categoricas_no_sup)

# Convertir variables categóricas a variables dummy
X_no_supervisado_preparado = pd.get_dummies(
    X_no_supervisado_preparado,
    columns=columnas_categoricas_no_sup,
    drop_first=True
)

print("\nDimensiones después de convertir variables categóricas:")
print(X_no_supervisado_preparado.shape)

# Convertir todo a numérico por seguridad
for col in X_no_supervisado_preparado.columns:
    X_no_supervisado_preparado[col] = pd.to_numeric(
        X_no_supervisado_preparado[col],
        errors="coerce"
    )

# Imputar nulos con la mediana
nulos_antes = X_no_supervisado_preparado.isnull().sum().sum()

X_no_supervisado_preparado = X_no_supervisado_preparado.fillna(
    X_no_supervisado_preparado.median()
)

nulos_despues = X_no_supervisado_preparado.isnull().sum().sum()

print("\nNulos antes de imputar:", nulos_antes)
print("Nulos después de imputar:", nulos_despues)

# Escalado de variables
scaler_no_supervisado = StandardScaler()

X_no_supervisado_scaled = scaler_no_supervisado.fit_transform(
    X_no_supervisado_preparado
)

print("\nDatos no supervisados escalados correctamente.")
print("Dimensiones finales de X_no_supervisado_scaled:")
print(X_no_supervisado_scaled.shape)

PREPARANDO DATOS PARA MODELOS NO SUPERVISADOS

Columnas numéricas:
['transaction_amount', 'monthly_income', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count']

Columnas categóricas:
['product_type', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'income_frequency', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel']

Dimensiones después de convertir variables categóricas:
(500000, 64)

Nulos antes de imputar: 0
Nulos después de imputar: 0

Datos no supervisados escalados correctamente.
Dimensiones finales de X_no_supervisado_scaled:
(500000, 64)


In [153]:
#Este bloque entrena un modelo Isolation Forest para detectar transacciones anómalas dentro del dataset válido. 
# El modelo analiza las variables preparadas en X_no_supervisado_scaled y asigna una marca de anomalía a cada registro.
# Se usa porque en fraude bancario no todos los patrones sospechosos están previamente etiquetados. 
# Isolation Forest permite identificar operaciones que se comportan diferente al resto sin utilizar la variable fraud_label durante el entrenamiento.

print("ENTRENANDO MODELO NO SUPERVISADO: ISOLATION FOREST")

# Validación de seguridad
if "X_no_supervisado_scaled" not in globals():
    raise NameError("X_no_supervisado_scaled no está definido. Ejecuta primero el bloque de preparación no supervisada.")

# Crear el modelo
modelo_isolation = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

# Entrenar el modelo
modelo_isolation.fit(X_no_supervisado_scaled)

# Predicciones originales del modelo
# Isolation Forest devuelve:
#  1 = normal
# -1 = anomalía
pred_isolation_original = modelo_isolation.predict(X_no_supervisado_scaled)

# Convertir al formato bancario:
# 0 = normal
# 1 = anomalía / sospechosa
pred_isolation = np.where(pred_isolation_original == -1, 1, 0)

# Score de anomalía
# Valores más bajos indican mayor anomalía
score_isolation = modelo_isolation.decision_function(X_no_supervisado_scaled)

# Guardar resultados en df_model
df_model["isolation_anomaly"] = pred_isolation
df_model["isolation_score"] = score_isolation

print("Modelo Isolation Forest entrenado correctamente.")
print("\nDistribución de anomalías detectadas:")
print(df_model["isolation_anomaly"].value_counts())

print("\nDistribución porcentual:")
print((df_model["isolation_anomaly"].value_counts(normalize=True) * 100).round(2))

ENTRENANDO MODELO NO SUPERVISADO: ISOLATION FOREST
Modelo Isolation Forest entrenado correctamente.

Distribución de anomalías detectadas:
isolation_anomaly
0    434203
1     65797
Name: count, dtype: int64

Distribución porcentual:
isolation_anomaly
0    86.84
1    13.16
Name: proportion, dtype: float64


In [154]:

# Este bloque compara las anomalías detectadas por Isolation Forest contra fraud_label.
# No entrena el modelo con fraud_label; solo evalúa si las anomalías coinciden con fraudes reales.
# EVALUACIÓN AUXILIAR DE ISOLATION FOREST

print("EVALUACIÓN AUXILIAR DE ISOLATION FOREST")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "isolation_anomaly" not in df_model.columns:
    raise KeyError("La columna isolation_anomaly no existe. Ejecuta primero el bloque de Isolation Forest.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Variable real y predicción del modelo no supervisado
y_real_iso = df_model[target].astype(int)
y_pred_iso = df_model["isolation_anomaly"].astype(int)

# Matriz de confusión
print("\nMatriz de confusión - Isolation Forest:")
matriz_iso = confusion_matrix(y_real_iso, y_pred_iso)
print(matriz_iso)

tn, fp, fn, tp = matriz_iso.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - Isolation Forest:")
print(classification_report(y_real_iso, y_pred_iso, zero_division=0))

# Métricas principales
accuracy_iso = accuracy_score(y_real_iso, y_pred_iso)
precision_iso = precision_score(y_real_iso, y_pred_iso, zero_division=0)
recall_iso = recall_score(y_real_iso, y_pred_iso, zero_division=0)
f1_iso = f1_score(y_real_iso, y_pred_iso, zero_division=0)

try:
    roc_auc_iso = roc_auc_score(y_real_iso, y_pred_iso)
except ValueError:
    roc_auc_iso = np.nan

print("\nMétricas principales - Isolation Forest:")
print(f"Accuracy : {accuracy_iso:.4f}")
print(f"Precision: {precision_iso:.4f}")
print(f"Recall   : {recall_iso:.4f}")
print(f"F1 Score : {f1_iso:.4f}")
print(f"ROC AUC  : {roc_auc_iso:.4f}")

# Tasa de fraude dentro de normales vs anomalías
analisis_iso = df_model.groupby("isolation_anomaly")[target].agg(
    cantidad="count",
    fraudes="sum",
    tasa_fraude="mean"
)

analisis_iso["tasa_fraude"] = (analisis_iso["tasa_fraude"] * 100).round(2)

print("\nTasa de fraude según clasificación de Isolation Forest:")
print(analisis_iso)

# Guardar resultados para comparación posterior
if "resultados_no_supervisados" not in globals():
    resultados_no_supervisados = []

resultados_no_supervisados.append({
    "Modelo": "Isolation Forest",
    "Accuracy": accuracy_iso,
    "Precision": precision_iso,
    "Recall": recall_iso,
    "F1 Score": f1_iso,
    "ROC AUC": roc_auc_iso,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_iso.sum())
})

print("\nResultados de Isolation Forest guardados correctamente.")

EVALUACIÓN AUXILIAR DE ISOLATION FOREST

Matriz de confusión - Isolation Forest:
[[396149  53851]
 [ 38054  11946]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 396,149
Falsos positivos      (FP): 53,851
Falsos negativos      (FN): 38,054
Verdaderos positivos  (TP): 11,946

Reporte de clasificación - Isolation Forest:
              precision    recall  f1-score   support

           0       0.91      0.88      0.90    450000
           1       0.18      0.24      0.21     50000

    accuracy                           0.82    500000
   macro avg       0.55      0.56      0.55    500000
weighted avg       0.84      0.82      0.83    500000


Métricas principales - Isolation Forest:
Accuracy : 0.8162
Precision: 0.1816
Recall   : 0.2389
F1 Score : 0.2063
ROC AUC  : 0.5596

Tasa de fraude según clasificación de Isolation Forest:
                   cantidad  fraudes  tasa_fraude
isolation_anomaly                                
0                    434203    38054         8.76
1

In [155]:
# Este bloque entrena K-Means para agrupar transacciones según patrones de comportamiento.
# K-Means no predice fraude directamente; crea grupos o clusters de transacciones similares.
# MODELO NO SUPERVISADO 2: K-MEANS


print("ENTRENANDO MODELO NO SUPERVISADO: K-MEANS")

# Validación de seguridad
if "X_no_supervisado_scaled" not in globals():
    raise NameError("X_no_supervisado_scaled no está definido. Ejecuta primero el bloque de preparación no supervisada.")

if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

# Definir cantidad de clusters
numero_clusters = 3

# Crear modelo K-Means
modelo_kmeans = KMeans(
    n_clusters=numero_clusters,
    random_state=42,
    n_init=10
)

# Entrenar modelo y asignar clusters
clusters_kmeans = modelo_kmeans.fit_predict(X_no_supervisado_scaled)

# Guardar cluster asignado en df_model
df_model["kmeans_cluster"] = clusters_kmeans

print("Modelo K-Means entrenado correctamente.")

print("\nDistribución de transacciones por cluster:")
print(df_model["kmeans_cluster"].value_counts().sort_index())

print("\nDistribución porcentual de transacciones por cluster:")
print((df_model["kmeans_cluster"].value_counts(normalize=True).sort_index() * 100).round(2))

# Calcular distancia de cada transacción al centroide de su cluster
distancias_centroides = modelo_kmeans.transform(X_no_supervisado_scaled)
df_model["kmeans_distance"] = np.min(distancias_centroides, axis=1)

print("\nDistancia al centroide calculada correctamente.")
print("Mientras mayor sea la distancia, más alejada está la transacción del comportamiento típico de su grupo.")

print("\nPrimeras filas con cluster y distancia:")
print(df_model[["kmeans_cluster", "kmeans_distance"]].head())

ENTRENANDO MODELO NO SUPERVISADO: K-MEANS
Modelo K-Means entrenado correctamente.

Distribución de transacciones por cluster:
kmeans_cluster
0     99301
1    151205
2    249494
Name: count, dtype: int64

Distribución porcentual de transacciones por cluster:
kmeans_cluster
0    19.86
1    30.24
2    49.90
Name: proportion, dtype: float64

Distancia al centroide calculada correctamente.
Mientras mayor sea la distancia, más alejada está la transacción del comportamiento típico de su grupo.

Primeras filas con cluster y distancia:
   kmeans_cluster  kmeans_distance
0               1         7.356031
1               1         8.628129
2               2         6.646800
3               1         7.872794
4               1         7.595764


In [156]:
# Este bloque evalúa los clusters creados por K-Means comparándolos contra fraud_label.
# K-Means no predice fraude directamente, por eso primero se identifica cuál cluster tiene mayor tasa de fraude.

# EVALUACIÓN AUXILIAR DE K-MEANS


print("EVALUACIÓN AUXILIAR DE K-MEANS")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "kmeans_cluster" not in df_model.columns:
    raise KeyError("La columna kmeans_cluster no existe. Ejecuta primero el bloque de K-Means.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Analizar fraude por cluster
fraude_por_cluster = df_model.groupby("kmeans_cluster")[target].agg(
    cantidad="count",
    fraudes="sum",
    tasa_fraude="mean"
)

fraude_por_cluster["tasa_fraude"] = (
    fraude_por_cluster["tasa_fraude"] * 100
).round(2)

print("\nFraude por cluster:")
print(fraude_por_cluster)

# Identificar el cluster con mayor tasa de fraude
cluster_mas_riesgoso = fraude_por_cluster["tasa_fraude"].idxmax()

print("\nCluster con mayor tasa de fraude:")
print(cluster_mas_riesgoso)

# Crear variable binaria para evaluar K-Means como señal de riesgo
# 0 = no pertenece al cluster más riesgoso
# 1 = pertenece al cluster más riesgoso
df_model["kmeans_cluster_riesgoso"] = np.where(
    df_model["kmeans_cluster"] == cluster_mas_riesgoso,
    1,
    0
)

# Variables para evaluación auxiliar
y_real_kmeans = df_model[target].astype(int)
y_pred_kmeans = df_model["kmeans_cluster_riesgoso"].astype(int)

# Matriz de confusión
print("\nMatriz de confusión - K-Means:")
matriz_kmeans = confusion_matrix(y_real_kmeans, y_pred_kmeans)
print(matriz_kmeans)

tn, fp, fn, tp = matriz_kmeans.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - K-Means:")
print(classification_report(y_real_kmeans, y_pred_kmeans, zero_division=0))

# Métricas principales
accuracy_kmeans = accuracy_score(y_real_kmeans, y_pred_kmeans)
precision_kmeans = precision_score(y_real_kmeans, y_pred_kmeans, zero_division=0)
recall_kmeans = recall_score(y_real_kmeans, y_pred_kmeans, zero_division=0)
f1_kmeans = f1_score(y_real_kmeans, y_pred_kmeans, zero_division=0)

try:
    roc_auc_kmeans = roc_auc_score(y_real_kmeans, y_pred_kmeans)
except ValueError:
    roc_auc_kmeans = np.nan

print("\nMétricas principales - K-Means:")
print(f"Accuracy : {accuracy_kmeans:.4f}")
print(f"Precision: {precision_kmeans:.4f}")
print(f"Recall   : {recall_kmeans:.4f}")
print(f"F1 Score : {f1_kmeans:.4f}")
print(f"ROC AUC  : {roc_auc_kmeans:.4f}")

# Guardar resultados para comparación posterior
if "resultados_no_supervisados" not in globals():
    resultados_no_supervisados = []

resultados_no_supervisados.append({
    "Modelo": "K-Means Cluster Riesgoso",
    "Accuracy": accuracy_kmeans,
    "Precision": precision_kmeans,
    "Recall": recall_kmeans,
    "F1 Score": f1_kmeans,
    "ROC AUC": roc_auc_kmeans,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_kmeans.sum())
})

print("\nResultados de K-Means guardados correctamente.")

EVALUACIÓN AUXILIAR DE K-MEANS

Fraude por cluster:
                cantidad  fraudes  tasa_fraude
kmeans_cluster                                
0                  99301     8550         8.61
1                 151205    22804        15.08
2                 249494    18646         7.47

Cluster con mayor tasa de fraude:
1

Matriz de confusión - K-Means:
[[321599 128401]
 [ 27196  22804]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 321,599
Falsos positivos      (FP): 128,401
Falsos negativos      (FN): 27,196
Verdaderos positivos  (TP): 22,804

Reporte de clasificación - K-Means:
              precision    recall  f1-score   support

           0       0.92      0.71      0.81    450000
           1       0.15      0.46      0.23     50000

    accuracy                           0.69    500000
   macro avg       0.54      0.59      0.52    500000
weighted avg       0.84      0.69      0.75    500000


Métricas principales - K-Means:
Accuracy : 0.6888
Precision: 0.1508
Reca

In [157]:
# Este bloque evalúa los clusters creados por K-Means comparándolos contra fraud_label.
# K-Means no predice fraude directamente, por eso primero se identifica cuál cluster tiene mayor tasa de fraude.

# EVALUACIÓN AUXILIAR DE K-MEANS


print("EVALUACIÓN AUXILIAR DE K-MEANS")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "kmeans_cluster" not in df_model.columns:
    raise KeyError("La columna kmeans_cluster no existe. Ejecuta primero el bloque de K-Means.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Analizar fraude por cluster
fraude_por_cluster = df_model.groupby("kmeans_cluster")[target].agg(
    cantidad="count",
    fraudes="sum",
    tasa_fraude="mean"
)

fraude_por_cluster["tasa_fraude"] = (
    fraude_por_cluster["tasa_fraude"] * 100
).round(2)

print("\nFraude por cluster:")
print(fraude_por_cluster)

# Identificar el cluster con mayor tasa de fraude
cluster_mas_riesgoso = fraude_por_cluster["tasa_fraude"].idxmax()

print("\nCluster con mayor tasa de fraude:")
print(cluster_mas_riesgoso)

# Crear variable binaria para evaluar K-Means como señal de riesgo
# 0 = no pertenece al cluster más riesgoso
# 1 = pertenece al cluster más riesgoso
df_model["kmeans_cluster_riesgoso"] = np.where(
    df_model["kmeans_cluster"] == cluster_mas_riesgoso,
    1,
    0
)

# Variables para evaluación auxiliar
y_real_kmeans = df_model[target].astype(int)
y_pred_kmeans = df_model["kmeans_cluster_riesgoso"].astype(int)

# Matriz de confusión
print("\nMatriz de confusión - K-Means:")
matriz_kmeans = confusion_matrix(y_real_kmeans, y_pred_kmeans)
print(matriz_kmeans)

tn, fp, fn, tp = matriz_kmeans.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - K-Means:")
print(classification_report(y_real_kmeans, y_pred_kmeans, zero_division=0))

# Métricas principales
accuracy_kmeans = accuracy_score(y_real_kmeans, y_pred_kmeans)
precision_kmeans = precision_score(y_real_kmeans, y_pred_kmeans, zero_division=0)
recall_kmeans = recall_score(y_real_kmeans, y_pred_kmeans, zero_division=0)
f1_kmeans = f1_score(y_real_kmeans, y_pred_kmeans, zero_division=0)

try:
    roc_auc_kmeans = roc_auc_score(y_real_kmeans, y_pred_kmeans)
except ValueError:
    roc_auc_kmeans = np.nan

print("\nMétricas principales - K-Means:")
print(f"Accuracy : {accuracy_kmeans:.4f}")
print(f"Precision: {precision_kmeans:.4f}")
print(f"Recall   : {recall_kmeans:.4f}")
print(f"F1 Score : {f1_kmeans:.4f}")
print(f"ROC AUC  : {roc_auc_kmeans:.4f}")

# Guardar resultados para comparación posterior
if "resultados_no_supervisados" not in globals():
    resultados_no_supervisados = []

resultados_no_supervisados.append({
    "Modelo": "K-Means Cluster Riesgoso",
    "Accuracy": accuracy_kmeans,
    "Precision": precision_kmeans,
    "Recall": recall_kmeans,
    "F1 Score": f1_kmeans,
    "ROC AUC": roc_auc_kmeans,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_kmeans.sum())
})

print("\nResultados de K-Means guardados correctamente.")

EVALUACIÓN AUXILIAR DE K-MEANS

Fraude por cluster:
                cantidad  fraudes  tasa_fraude
kmeans_cluster                                
0                  99301     8550         8.61
1                 151205    22804        15.08
2                 249494    18646         7.47

Cluster con mayor tasa de fraude:
1

Matriz de confusión - K-Means:
[[321599 128401]
 [ 27196  22804]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 321,599
Falsos positivos      (FP): 128,401
Falsos negativos      (FN): 27,196
Verdaderos positivos  (TP): 22,804

Reporte de clasificación - K-Means:
              precision    recall  f1-score   support

           0       0.92      0.71      0.81    450000
           1       0.15      0.46      0.23     50000

    accuracy                           0.69    500000
   macro avg       0.54      0.59      0.52    500000
weighted avg       0.84      0.69      0.75    500000


Métricas principales - K-Means:
Accuracy : 0.6888
Precision: 0.1508
Reca

In [158]:
# Este bloque combina las señales de Isolation Forest y K-Means para crear un score no supervisado de riesgo.
# El objetivo es clasificar las transacciones en bajo riesgo, riesgo medio o alto riesgo según las alertas generadas por ambos modelos.

# MODELO COMBINADO NO SUPERVISADO


print("MODELO COMBINADO NO SUPERVISADO: ISOLATION FOREST + K-MEANS")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "isolation_anomaly" not in df_model.columns:
    raise KeyError("La columna isolation_anomaly no existe. Ejecuta primero el bloque de Isolation Forest.")

if "kmeans_cluster_riesgoso" not in df_model.columns:
    raise KeyError("La columna kmeans_cluster_riesgoso no existe. Ejecuta primero la evaluación auxiliar de K-Means.")

# Crear score combinado
df_model["score_no_supervisado"] = 0

# Señal 1: anomalía detectada por Isolation Forest
df_model.loc[
    df_model["isolation_anomaly"] == 1,
    "score_no_supervisado"
] += 1

# Señal 2: pertenece al cluster más riesgoso de K-Means
df_model.loc[
    df_model["kmeans_cluster_riesgoso"] == 1,
    "score_no_supervisado"
] += 1

# Clasificación de riesgo
df_model["riesgo_no_supervisado"] = np.select(
    [
        df_model["score_no_supervisado"] == 0,
        df_model["score_no_supervisado"] == 1,
        df_model["score_no_supervisado"] >= 2
    ],
    [
        "bajo riesgo",
        "riesgo medio",
        "alto riesgo"
    ],
    default="sin clasificar"
)

# Predicción binaria para evaluación auxiliar
# 0 = no sospechosa
# 1 = sospechosa
df_model["pred_no_supervisado_combinado"] = np.where(
    df_model["score_no_supervisado"] >= 1,
    1,
    0
)

print("\nDistribución del score no supervisado:")
print(df_model["score_no_supervisado"].value_counts().sort_index())

print("\nDistribución del riesgo no supervisado:")
print(df_model["riesgo_no_supervisado"].value_counts())

print("\nPrimeras filas del score combinado:")
print(
    df_model[
        [
            "isolation_anomaly",
            "kmeans_cluster_riesgoso",
            "score_no_supervisado",
            "riesgo_no_supervisado",
            "pred_no_supervisado_combinado"
        ]
    ].head()
)

MODELO COMBINADO NO SUPERVISADO: ISOLATION FOREST + K-MEANS

Distribución del score no supervisado:
score_no_supervisado
0    296317
1    190364
2     13319
Name: count, dtype: int64

Distribución del riesgo no supervisado:
riesgo_no_supervisado
bajo riesgo     296317
riesgo medio    190364
alto riesgo      13319
Name: count, dtype: int64

Primeras filas del score combinado:
   isolation_anomaly  kmeans_cluster_riesgoso  score_no_supervisado  \
0                  0                        1                     1   
1                  1                        1                     2   
2                  0                        0                     0   
3                  0                        1                     1   
4                  0                        1                     1   

  riesgo_no_supervisado  pred_no_supervisado_combinado  
0          riesgo medio                              1  
1           alto riesgo                              1  
2           bajo riesgo 

In [159]:
# Este bloque evalúa el modelo combinado no supervisado contra fraud_label.
# La evaluación permite saber si las señales combinadas de Isolation Forest y K-Means ayudan a identificar fraudes reales.

# EVALUACIÓN AUXILIAR DEL MODELO COMBINADO


print("EVALUACIÓN AUXILIAR DEL MODELO COMBINADO NO SUPERVISADO")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "pred_no_supervisado_combinado" not in df_model.columns:
    raise KeyError("La columna pred_no_supervisado_combinado no existe. Ejecuta primero el bloque del modelo combinado.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Variable real y predicción combinada
y_real_comb = df_model[target].astype(int)
y_pred_comb = df_model["pred_no_supervisado_combinado"].astype(int)

# Matriz de confusión
print("\nMatriz de confusión - Modelo combinado no supervisado:")
matriz_comb = confusion_matrix(y_real_comb, y_pred_comb)
print(matriz_comb)

tn, fp, fn, tp = matriz_comb.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - Modelo combinado no supervisado:")
print(classification_report(y_real_comb, y_pred_comb, zero_division=0))

# Métricas principales
accuracy_comb = accuracy_score(y_real_comb, y_pred_comb)
precision_comb = precision_score(y_real_comb, y_pred_comb, zero_division=0)
recall_comb = recall_score(y_real_comb, y_pred_comb, zero_division=0)
f1_comb = f1_score(y_real_comb, y_pred_comb, zero_division=0)

try:
    roc_auc_comb = roc_auc_score(y_real_comb, y_pred_comb)
except ValueError:
    roc_auc_comb = np.nan

print("\nMétricas principales - Modelo combinado no supervisado:")
print(f"Accuracy : {accuracy_comb:.4f}")
print(f"Precision: {precision_comb:.4f}")
print(f"Recall   : {recall_comb:.4f}")
print(f"F1 Score : {f1_comb:.4f}")
print(f"ROC AUC  : {roc_auc_comb:.4f}")

# Análisis de fraude por score
fraude_por_score = df_model.groupby("score_no_supervisado")[target].agg(
    cantidad="count",
    fraudes="sum",
    tasa_fraude="mean"
)

fraude_por_score["tasa_fraude"] = (
    fraude_por_score["tasa_fraude"] * 100
).round(2)

print("\nFraude por score_no_supervisado:")
print(fraude_por_score)

# Análisis de fraude por nivel de riesgo
fraude_por_riesgo = df_model.groupby("riesgo_no_supervisado")[target].agg(
    cantidad="count",
    fraudes="sum",
    tasa_fraude="mean"
)

fraude_por_riesgo["tasa_fraude"] = (
    fraude_por_riesgo["tasa_fraude"] * 100
).round(2)

print("\nFraude por riesgo_no_supervisado:")
print(fraude_por_riesgo)

# Guardar resultados para comparación posterior
if "resultados_no_supervisados" not in globals():
    resultados_no_supervisados = []

resultados_no_supervisados.append({
    "Modelo": "Combinado No Supervisado",
    "Accuracy": accuracy_comb,
    "Precision": precision_comb,
    "Recall": recall_comb,
    "F1 Score": f1_comb,
    "ROC AUC": roc_auc_comb,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_comb.sum())
})

print("\nResultados del modelo combinado guardados correctamente.")

EVALUACIÓN AUXILIAR DEL MODELO COMBINADO NO SUPERVISADO

Matriz de confusión - Modelo combinado no supervisado:
[[277615 172385]
 [ 18702  31298]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 277,615
Falsos positivos      (FP): 172,385
Falsos negativos      (FN): 18,702
Verdaderos positivos  (TP): 31,298

Reporte de clasificación - Modelo combinado no supervisado:
              precision    recall  f1-score   support

           0       0.94      0.62      0.74    450000
           1       0.15      0.63      0.25     50000

    accuracy                           0.62    500000
   macro avg       0.55      0.62      0.50    500000
weighted avg       0.86      0.62      0.69    500000


Métricas principales - Modelo combinado no supervisado:
Accuracy : 0.6178
Precision: 0.1537
Recall   : 0.6260
F1 Score : 0.2467
ROC AUC  : 0.6214

Fraude por score_no_supervisado:
                      cantidad  fraudes  tasa_fraude
score_no_supervisado                                
0     

In [160]:
# Este bloque compara los resultados de los modelos no supervisados evaluados previamente.
# Permite revisar cuál modelo tuvo mejor desempeño auxiliar contra fraud_label antes de pasar a los modelos supervisados.

# COMPARACIÓN DE MODELOS NO SUPERVISADOS

print("COMPARACIÓN DE MODELOS NO SUPERVISADOS")

# Validación de seguridad
if "resultados_no_supervisados" not in globals():
    raise NameError("La lista resultados_no_supervisados no existe. Ejecuta primero las evaluaciones no supervisadas.")

# Crear DataFrame de resultados
df_resultados_no_supervisados = pd.DataFrame(resultados_no_supervisados)

# Ordenar por Recall, métrica clave en fraude
df_resultados_no_supervisados = df_resultados_no_supervisados.sort_values(
    by="Recall",
    ascending=False
)

print("\nResultados de modelos no supervisados ordenados por Recall:")
display(df_resultados_no_supervisados)

# Identificar mejor modelo según Recall
mejor_no_supervisado_recall = df_resultados_no_supervisados.iloc[0]

print("\nMejor modelo no supervisado según Recall:")
print(mejor_no_supervisado_recall["Modelo"])

print("\nRecall obtenido:")
print(round(mejor_no_supervisado_recall["Recall"], 4))

# Ordenar también por F1 Score para revisar equilibrio
df_no_sup_f1 = df_resultados_no_supervisados.sort_values(
    by="F1 Score",
    ascending=False
)

print("\nResultados ordenados por F1 Score:")
display(df_no_sup_f1)

print("\nConclusión parcial:")
print("Los modelos no supervisados fueron evaluados de forma auxiliar contra fraud_label.")
print("Estos modelos no fueron entrenados con la etiqueta de fraude.")
print("A partir del siguiente bloque inicia la etapa de modelos supervisados.")

COMPARACIÓN DE MODELOS NO SUPERVISADOS

Resultados de modelos no supervisados ordenados por Recall:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
3,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
7,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
11,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
1,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
10,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
2,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
6,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
5,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
9,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
0,Isolation Forest,0.816190,0.181558,0.23892,0.206327,0.559626,11946,53851,38054,65797



Mejor modelo no supervisado según Recall:
Combinado No Supervisado

Recall obtenido:
0.626

Resultados ordenados por F1 Score:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
3,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
7,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
11,Combinado No Supervisado,0.617826,0.153660,0.62596,0.246749,0.621441,31298,172385,18702,203683
1,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
10,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
2,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
6,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
5,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
9,K-Means Cluster Riesgoso,0.688806,0.150815,0.45608,0.226674,0.585372,22804,128401,27196,151205
0,Isolation Forest,0.816190,0.181558,0.23892,0.206327,0.559626,11946,53851,38054,65797



Conclusión parcial:
Los modelos no supervisados fueron evaluados de forma auxiliar contra fraud_label.
Estos modelos no fueron entrenados con la etiqueta de fraude.
A partir del siguiente bloque inicia la etapa de modelos supervisados.


In [161]:
# Este bloque define las variables X e y para los modelos supervisados.
# X_supervisado contiene las variables predictoras y y_supervisado contiene la etiqueta real fraud_label.

# DEFINIR X E y PARA MODELOS SUPERVISADOS


print("DEFINIENDO X E y PARA MODELOS SUPERVISADOS")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "columnas_predictoras_supervisado" not in globals():
    raise NameError("columnas_predictoras_supervisado no está definida. Ejecuta primero el bloque de columnas predictoras.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Definir variables predictoras y variable objetivo
X_supervisado = df_model[columnas_predictoras_supervisado].copy()
y_supervisado = df_model[target].copy()

print("Dimensiones de X_supervisado:")
print(X_supervisado.shape)

print("\nDimensiones de y_supervisado:")
print(y_supervisado.shape)

print("\nDistribución de y_supervisado:")
print(y_supervisado.value_counts())

print("\nDistribución porcentual de y_supervisado:")
print((y_supervisado.value_counts(normalize=True) * 100).round(2))

DEFINIENDO X E y PARA MODELOS SUPERVISADOS
Dimensiones de X_supervisado:
(500000, 20)

Dimensiones de y_supervisado:
(500000,)

Distribución de y_supervisado:
fraud_label
0    450000
1     50000
Name: count, dtype: int64

Distribución porcentual de y_supervisado:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64


In [162]:
# Este bloque define las variables X e y para los modelos supervisados.
# X_supervisado contiene las variables predictoras y y_supervisado contiene la etiqueta real fraud_label.

# DEFINIR X E y PARA MODELOS SUPERVISADOS

print("DEFINIENDO X E y PARA MODELOS SUPERVISADOS")

# Validaciones de seguridad
if "df_model" not in globals():
    raise NameError("df_model no está definido. Ejecuta primero el bloque de filtrado de datos válidos.")

if "columnas_predictoras_supervisado" not in globals():
    raise NameError("columnas_predictoras_supervisado no está definida. Ejecuta primero el bloque de columnas predictoras.")

if target not in df_model.columns:
    raise KeyError(f"La variable objetivo {target} no existe en df_model.")

# Definir variables predictoras y variable objetivo
X_supervisado = df_model[columnas_predictoras_supervisado].copy()
y_supervisado = df_model[target].copy()

print("Dimensiones de X_supervisado:")
print(X_supervisado.shape)

print("\nDimensiones de y_supervisado:")
print(y_supervisado.shape)

print("\nDistribución de y_supervisado:")
print(y_supervisado.value_counts())

print("\nDistribución porcentual de y_supervisado:")
print((y_supervisado.value_counts(normalize=True) * 100).round(2))

DEFINIENDO X E y PARA MODELOS SUPERVISADOS
Dimensiones de X_supervisado:
(500000, 20)

Dimensiones de y_supervisado:
(500000,)

Distribución de y_supervisado:
fraud_label
0    450000
1     50000
Name: count, dtype: int64

Distribución porcentual de y_supervisado:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64


In [163]:
# Este bloque prepara X_supervisado para los modelos supervisados.
# Convierte variables categóricas en variables numéricas, imputa valores nulos y deja el dataset listo para dividirlo en entrenamiento y prueba.

# PREPARACIÓN DE DATOS PARA MODELOS SUPERVISADOS


print("PREPARANDO DATOS PARA MODELOS SUPERVISADOS")

# Validaciones de seguridad
if "X_supervisado" not in globals():
    raise NameError("X_supervisado no está definido. Ejecuta primero el bloque de definición de X e y supervisados.")

if "y_supervisado" not in globals():
    raise NameError("y_supervisado no está definido. Ejecuta primero el bloque de definición de X e y supervisados.")

# Crear copia de trabajo
X_supervisado_preparado = X_supervisado.copy()

# Identificar columnas categóricas y numéricas
columnas_categoricas_sup = X_supervisado_preparado.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

columnas_numericas_sup = X_supervisado_preparado.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("\nColumnas numéricas:")
print(columnas_numericas_sup)

print("\nColumnas categóricas:")
print(columnas_categoricas_sup)

# Convertir variables categóricas a variables dummy
X_supervisado_preparado = pd.get_dummies(
    X_supervisado_preparado,
    columns=columnas_categoricas_sup,
    drop_first=True
)

print("\nDimensiones después de convertir variables categóricas:")
print(X_supervisado_preparado.shape)

# Convertir todo a numérico por seguridad
for col in X_supervisado_preparado.columns:
    X_supervisado_preparado[col] = pd.to_numeric(
        X_supervisado_preparado[col],
        errors="coerce"
    )

# Imputar nulos con la mediana
nulos_antes_sup = X_supervisado_preparado.isnull().sum().sum()

X_supervisado_preparado = X_supervisado_preparado.fillna(
    X_supervisado_preparado.median()
)

nulos_despues_sup = X_supervisado_preparado.isnull().sum().sum()

print("\nNulos antes de imputar:", nulos_antes_sup)
print("Nulos después de imputar:", nulos_despues_sup)

print("\nDatos supervisados preparados correctamente.")
print("Dimensiones finales de X_supervisado_preparado:")
print(X_supervisado_preparado.shape)

PREPARANDO DATOS PARA MODELOS SUPERVISADOS

Columnas numéricas:
['transaction_amount', 'monthly_income', 'customer_tenure_months', 'avg_3m_transaction_amount', 'avg_3m_transaction_count']

Columnas categóricas:
['product_type', 'transaction_direction', 'channel', 'transaction_status', 'merchant_or_destination', 'source_account_type', 'destination_account_type', 'customer_segment', 'occupation', 'industry', 'employment_type', 'income_frequency', 'usual_transaction_hour_range', 'usual_product_type', 'usual_channel']

Dimensiones después de convertir variables categóricas:
(500000, 64)

Nulos antes de imputar: 0
Nulos después de imputar: 0

Datos supervisados preparados correctamente.
Dimensiones finales de X_supervisado_preparado:
(500000, 64)


In [164]:
# Este bloque divide los datos supervisados en entrenamiento y prueba.
# La división permite entrenar los modelos con una parte de los datos y evaluarlos con datos no vistos.

# DIVISIÓN TRAIN / TEST PARA MODELOS SUPERVISADOS


print("DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA")

# Validaciones de seguridad
if "X_supervisado_preparado" not in globals():
    raise NameError("X_supervisado_preparado no está definido. Ejecuta primero el bloque de preparación supervisada.")

if "y_supervisado" not in globals():
    raise NameError("y_supervisado no está definido. Ejecuta primero el bloque de definición de X e y supervisados.")

# División de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_supervisado_preparado,
    y_supervisado,
    test_size=0.30,
    random_state=42,
    stratify=y_supervisado
)

print("Dimensiones de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nDimensiones de prueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nDistribución de y_train:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución de y_test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA
Dimensiones de entrenamiento:
X_train: (350000, 64)
y_train: (350000,)

Dimensiones de prueba:
X_test: (150000, 64)
y_test: (150000,)

Distribución de y_train:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64

Distribución de y_test:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64


In [165]:
# Este bloque divide los datos supervisados en entrenamiento y prueba.
# La división permite entrenar los modelos con una parte de los datos y evaluarlos con datos no vistos.

# DIVISIÓN TRAIN / TEST PARA MODELOS SUPERVISADOS


print("DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA")

# Validaciones de seguridad
if "X_supervisado_preparado" not in globals():
    raise NameError("X_supervisado_preparado no está definido. Ejecuta primero el bloque de preparación supervisada.")

if "y_supervisado" not in globals():
    raise NameError("y_supervisado no está definido. Ejecuta primero el bloque de definición de X e y supervisados.")

# División de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_supervisado_preparado,
    y_supervisado,
    test_size=0.30,
    random_state=42,
    stratify=y_supervisado
)

print("Dimensiones de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nDimensiones de prueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nDistribución de y_train:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución de y_test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA
Dimensiones de entrenamiento:
X_train: (350000, 64)
y_train: (350000,)

Dimensiones de prueba:
X_test: (150000, 64)
y_test: (150000,)

Distribución de y_train:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64

Distribución de y_test:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64


In [166]:
# Este bloque divide los datos supervisados en entrenamiento y prueba.
# La división permite entrenar los modelos con una parte de los datos y evaluarlos con datos no vistos.

# DIVISIÓN TRAIN / TEST PARA MODELOS SUPERVISADOS


print("DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA")

# Validaciones de seguridad
if "X_supervisado_preparado" not in globals():
    raise NameError("X_supervisado_preparado no está definido. Ejecuta primero el bloque de preparación supervisada.")

if "y_supervisado" not in globals():
    raise NameError("y_supervisado no está definido. Ejecuta primero el bloque de definición de X e y supervisados.")

# División de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_supervisado_preparado,
    y_supervisado,
    test_size=0.30,
    random_state=42,
    stratify=y_supervisado
)

print("Dimensiones de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nDimensiones de prueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nDistribución de y_train:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución de y_test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

DIVIDIENDO DATOS EN ENTRENAMIENTO Y PRUEBA
Dimensiones de entrenamiento:
X_train: (350000, 64)
y_train: (350000,)

Dimensiones de prueba:
X_test: (150000, 64)
y_test: (150000,)

Distribución de y_train:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64

Distribución de y_test:
fraud_label
0    90.0
1    10.0
Name: proportion, dtype: float64


In [167]:
# Este bloque escala las variables supervisadas usando StandardScaler.
# El escalador se ajusta solo con X_train para evitar fuga de información y luego se aplica a X_test.

# ==========================================
# ESCALADO DE DATOS PARA MODELOS SUPERVISADOS
# ==========================================

print("ESCALANDO DATOS PARA MODELOS SUPERVISADOS")

# Validaciones de seguridad
if "X_train" not in globals():
    raise NameError("X_train no está definido. Ejecuta primero el bloque de división train/test.")

if "X_test" not in globals():
    raise NameError("X_test no está definido. Ejecuta primero el bloque de división train/test.")

# Crear escalador
scaler_supervisado = StandardScaler()

# Ajustar solo con X_train y transformar X_train
X_train_scaled = scaler_supervisado.fit_transform(X_train)

# Transformar X_test con el mismo escalador
X_test_scaled = scaler_supervisado.transform(X_test)

print("Escalado completado correctamente.")

print("\nDimensiones de X_train_scaled:")
print(X_train_scaled.shape)

print("\nDimensiones de X_test_scaled:")
print(X_test_scaled.shape)

ESCALANDO DATOS PARA MODELOS SUPERVISADOS
Escalado completado correctamente.

Dimensiones de X_train_scaled:
(350000, 64)

Dimensiones de X_test_scaled:
(150000, 64)


In [168]:
# Este bloque entrena y evalúa el primer modelo supervisado: Logistic Regression.
# El modelo aprende con X_train_scaled y y_train, luego predice sobre X_test_scaled para medir su desempeño.

# ==========================================
# MODELO SUPERVISADO 1: LOGISTIC REGRESSION
# ==========================================

print("ENTRENANDO MODELO SUPERVISADO: LOGISTIC REGRESSION")

# Validaciones de seguridad
if "X_train_scaled" not in globals():
    raise NameError("X_train_scaled no está definido. Ejecuta primero el bloque de escalado supervisado.")

if "X_test_scaled" not in globals():
    raise NameError("X_test_scaled no está definido. Ejecuta primero el bloque de escalado supervisado.")

if "y_train" not in globals():
    raise NameError("y_train no está definido. Ejecuta primero el bloque de división train/test.")

if "y_test" not in globals():
    raise NameError("y_test no está definido. Ejecuta primero el bloque de división train/test.")

# Crear modelo
modelo_logistico = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

# Entrenar modelo
modelo_logistico.fit(X_train_scaled, y_train)

# Predicciones
y_pred_logistico = modelo_logistico.predict(X_test_scaled)

# Probabilidades para ROC AUC
y_proba_logistico = modelo_logistico.predict_proba(X_test_scaled)[:, 1]

print("Modelo Logistic Regression entrenado correctamente.")

# Matriz de confusión
print("\nMatriz de confusión - Logistic Regression:")
matriz_logistico = confusion_matrix(y_test, y_pred_logistico)
print(matriz_logistico)

tn, fp, fn, tp = matriz_logistico.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - Logistic Regression:")
print(classification_report(y_test, y_pred_logistico, zero_division=0))

# Métricas principales
accuracy_logistico = accuracy_score(y_test, y_pred_logistico)
precision_logistico = precision_score(y_test, y_pred_logistico, zero_division=0)
recall_logistico = recall_score(y_test, y_pred_logistico, zero_division=0)
f1_logistico = f1_score(y_test, y_pred_logistico, zero_division=0)
roc_auc_logistico = roc_auc_score(y_test, y_proba_logistico)

print("\nMétricas principales - Logistic Regression:")
print(f"Accuracy : {accuracy_logistico:.4f}")
print(f"Precision: {precision_logistico:.4f}")
print(f"Recall   : {recall_logistico:.4f}")
print(f"F1 Score : {f1_logistico:.4f}")
print(f"ROC AUC  : {roc_auc_logistico:.4f}")

# Guardar resultados para comparación posterior
if "resultados_supervisados" not in globals():
    resultados_supervisados = []

resultados_supervisados.append({
    "Modelo": "Logistic Regression",
    "Accuracy": accuracy_logistico,
    "Precision": precision_logistico,
    "Recall": recall_logistico,
    "F1 Score": f1_logistico,
    "ROC AUC": roc_auc_logistico,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_logistico.sum())
})

print("\nResultados de Logistic Regression guardados correctamente.")

ENTRENANDO MODELO SUPERVISADO: LOGISTIC REGRESSION
Modelo Logistic Regression entrenado correctamente.

Matriz de confusión - Logistic Regression:
[[134984     16]
 [     7  14993]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 134,984
Falsos positivos      (FP): 16
Falsos negativos      (FN): 7
Verdaderos positivos  (TP): 14,993

Reporte de clasificación - Logistic Regression:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    135000
           1       1.00      1.00      1.00     15000

    accuracy                           1.00    150000
   macro avg       1.00      1.00      1.00    150000
weighted avg       1.00      1.00      1.00    150000


Métricas principales - Logistic Regression:
Accuracy : 0.9998
Precision: 0.9989
Recall   : 0.9995
F1 Score : 0.9992
ROC AUC  : 1.0000

Resultados de Logistic Regression guardados correctamente.


In [169]:
# Este bloque entrena y evalúa el segundo modelo supervisado: Random Forest.
# Random Forest aprende múltiples árboles de decisión para clasificar transacciones regulares y fraudulentas.

# ==========================================
# MODELO SUPERVISADO 2: RANDOM FOREST
# ==========================================

print("ENTRENANDO MODELO SUPERVISADO: RANDOM FOREST")

# Validaciones de seguridad
if "X_train" not in globals():
    raise NameError("X_train no está definido. Ejecuta primero el bloque de división train/test.")

if "X_test" not in globals():
    raise NameError("X_test no está definido. Ejecuta primero el bloque de división train/test.")

if "y_train" not in globals():
    raise NameError("y_train no está definido. Ejecuta primero el bloque de división train/test.")

if "y_test" not in globals():
    raise NameError("y_test no está definido. Ejecuta primero el bloque de división train/test.")

# Crear modelo
modelo_random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Entrenar modelo
modelo_random_forest.fit(X_train, y_train)

# Predicciones
y_pred_rf = modelo_random_forest.predict(X_test)

# Probabilidades para ROC AUC
y_proba_rf = modelo_random_forest.predict_proba(X_test)[:, 1]

print("Modelo Random Forest entrenado correctamente.")

# Matriz de confusión
print("\nMatriz de confusión - Random Forest:")
matriz_rf = confusion_matrix(y_test, y_pred_rf)
print(matriz_rf)

tn, fp, fn, tp = matriz_rf.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# Métricas principales
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("\nMétricas principales - Random Forest:")
print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print(f"ROC AUC  : {roc_auc_rf:.4f}")

# Guardar resultados para comparación posterior
if "resultados_supervisados" not in globals():
    resultados_supervisados = []

resultados_supervisados.append({
    "Modelo": "Random Forest",
    "Accuracy": accuracy_rf,
    "Precision": precision_rf,
    "Recall": recall_rf,
    "F1 Score": f1_rf,
    "ROC AUC": roc_auc_rf,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_rf.sum())
})

print("\nResultados de Random Forest guardados correctamente.")

ENTRENANDO MODELO SUPERVISADO: RANDOM FOREST
Modelo Random Forest entrenado correctamente.

Matriz de confusión - Random Forest:
[[135000      0]
 [    15  14985]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 135,000
Falsos positivos      (FP): 0
Falsos negativos      (FN): 15
Verdaderos positivos  (TP): 14,985

Reporte de clasificación - Random Forest:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    135000
           1       1.00      1.00      1.00     15000

    accuracy                           1.00    150000
   macro avg       1.00      1.00      1.00    150000
weighted avg       1.00      1.00      1.00    150000


Métricas principales - Random Forest:
Accuracy : 0.9999
Precision: 1.0000
Recall   : 0.9990
F1 Score : 0.9995
ROC AUC  : 1.0000

Resultados de Random Forest guardados correctamente.


In [170]:
# Este bloque entrena y evalúa el segundo modelo supervisado: Random Forest.
# Random Forest aprende múltiples árboles de decisión para clasificar transacciones regulares y fraudulentas.

# ==========================================
# MODELO SUPERVISADO 2: RANDOM FOREST
# ==========================================

print("ENTRENANDO MODELO SUPERVISADO: RANDOM FOREST")

# Validaciones de seguridad
if "X_train" not in globals():
    raise NameError("X_train no está definido. Ejecuta primero el bloque de división train/test.")

if "X_test" not in globals():
    raise NameError("X_test no está definido. Ejecuta primero el bloque de división train/test.")

if "y_train" not in globals():
    raise NameError("y_train no está definido. Ejecuta primero el bloque de división train/test.")

if "y_test" not in globals():
    raise NameError("y_test no está definido. Ejecuta primero el bloque de división train/test.")

# Crear modelo
modelo_random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Entrenar modelo
modelo_random_forest.fit(X_train, y_train)

# Predicciones
y_pred_rf = modelo_random_forest.predict(X_test)

# Probabilidades para ROC AUC
y_proba_rf = modelo_random_forest.predict_proba(X_test)[:, 1]

print("Modelo Random Forest entrenado correctamente.")

# Matriz de confusión
print("\nMatriz de confusión - Random Forest:")
matriz_rf = confusion_matrix(y_test, y_pred_rf)
print(matriz_rf)

tn, fp, fn, tp = matriz_rf.ravel()

print("\nInterpretación de la matriz:")
print(f"Verdaderos negativos  (TN): {tn:,}")
print(f"Falsos positivos      (FP): {fp:,}")
print(f"Falsos negativos      (FN): {fn:,}")
print(f"Verdaderos positivos  (TP): {tp:,}")

# Reporte de clasificación
print("\nReporte de clasificación - Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# Métricas principales
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("\nMétricas principales - Random Forest:")
print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print(f"ROC AUC  : {roc_auc_rf:.4f}")

# Guardar resultados para comparación posterior
if "resultados_supervisados" not in globals():
    resultados_supervisados = []

resultados_supervisados.append({
    "Modelo": "Random Forest",
    "Accuracy": accuracy_rf,
    "Precision": precision_rf,
    "Recall": recall_rf,
    "F1 Score": f1_rf,
    "ROC AUC": roc_auc_rf,
    "Fraudes Detectados": int(tp),
    "Falsos Positivos": int(fp),
    "Falsos Negativos": int(fn),
    "Alertas Generadas": int(y_pred_rf.sum())
})

print("\nResultados de Random Forest guardados correctamente.")

ENTRENANDO MODELO SUPERVISADO: RANDOM FOREST
Modelo Random Forest entrenado correctamente.

Matriz de confusión - Random Forest:
[[135000      0]
 [    15  14985]]

Interpretación de la matriz:
Verdaderos negativos  (TN): 135,000
Falsos positivos      (FP): 0
Falsos negativos      (FN): 15
Verdaderos positivos  (TP): 14,985

Reporte de clasificación - Random Forest:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    135000
           1       1.00      1.00      1.00     15000

    accuracy                           1.00    150000
   macro avg       1.00      1.00      1.00    150000
weighted avg       1.00      1.00      1.00    150000


Métricas principales - Random Forest:
Accuracy : 0.9999
Precision: 1.0000
Recall   : 0.9990
F1 Score : 0.9995
ROC AUC  : 1.0000

Resultados de Random Forest guardados correctamente.


In [171]:
# Este bloque compara los resultados de los modelos supervisados entrenados previamente.
# Permite identificar cuál modelo tuvo mejor desempeño según Recall, F1 Score y ROC AUC.

# ==========================================
# COMPARACIÓN DE MODELOS SUPERVISADOS
# ==========================================

print("COMPARACIÓN DE MODELOS SUPERVISADOS")

# Validación de seguridad
if "resultados_supervisados" not in globals():
    raise NameError("La lista resultados_supervisados no existe. Ejecuta primero los modelos supervisados.")

# Crear DataFrame de resultados
df_resultados_supervisados = pd.DataFrame(resultados_supervisados)

print("\nResultados generales de modelos supervisados:")
display(df_resultados_supervisados)

# Ordenar por Recall
df_sup_recall = df_resultados_supervisados.sort_values(
    by="Recall",
    ascending=False
)

print("\nModelos supervisados ordenados por Recall:")
display(df_sup_recall)

# Ordenar por F1 Score
df_sup_f1 = df_resultados_supervisados.sort_values(
    by="F1 Score",
    ascending=False
)

print("\nModelos supervisados ordenados por F1 Score:")
display(df_sup_f1)

# Ordenar por ROC AUC
df_sup_auc = df_resultados_supervisados.sort_values(
    by="ROC AUC",
    ascending=False
)

print("\nModelos supervisados ordenados por ROC AUC:")
display(df_sup_auc)

# Identificar mejores modelos según métricas clave
mejor_modelo_recall = df_sup_recall.iloc[0]
mejor_modelo_f1 = df_sup_f1.iloc[0]
mejor_modelo_auc = df_sup_auc.iloc[0]

print("\nMejor modelo según Recall:")
print(mejor_modelo_recall["Modelo"], "-", round(mejor_modelo_recall["Recall"], 4))

print("\nMejor modelo según F1 Score:")
print(mejor_modelo_f1["Modelo"], "-", round(mejor_modelo_f1["F1 Score"], 4))

print("\nMejor modelo según ROC AUC:")
print(mejor_modelo_auc["Modelo"], "-", round(mejor_modelo_auc["ROC AUC"], 4))

print("\nConclusión parcial:")
print("En fraude bancario, Recall es clave porque mide cuántos fraudes reales fueron detectados.")
print("F1 Score ayuda a equilibrar Recall y Precision.")
print("ROC AUC mide la capacidad general del modelo para separar fraudes de no fraudes.")

COMPARACIÓN DE MODELOS SUPERVISADOS

Resultados generales de modelos supervisados:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
0,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
1,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
2,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
3,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
4,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
5,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
6,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
7,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
8,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985



Modelos supervisados ordenados por Recall:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
0,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
3,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
6,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
2,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
1,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
4,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
5,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
7,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
8,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985



Modelos supervisados ordenados por F1 Score:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
1,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
8,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
2,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
4,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
5,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
7,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
0,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
3,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
6,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009



Modelos supervisados ordenados por ROC AUC:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas
0,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
3,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
6,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.0,14993,16,7,15009
2,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
1,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
4,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
5,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
7,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985
8,Random Forest,0.999900,1.000000,0.999000,0.999500,1.0,14985,0,15,14985



Mejor modelo según Recall:
Logistic Regression - 0.9995

Mejor modelo según F1 Score:
Random Forest - 0.9995

Mejor modelo según ROC AUC:
Logistic Regression - 1.0

Conclusión parcial:
En fraude bancario, Recall es clave porque mide cuántos fraudes reales fueron detectados.
F1 Score ayuda a equilibrar Recall y Precision.
ROC AUC mide la capacidad general del modelo para separar fraudes de no fraudes.


In [172]:
# Este bloque compara todos los modelos evaluados en el proyecto.
# Une los resultados de modelos no supervisados y supervisados para revisar cuál tuvo mejor desempeño general.

# ==========================================
# COMPARACIÓN FINAL DE TODOS LOS MODELOS
# ==========================================

print("COMPARACIÓN FINAL DE TODOS LOS MODELOS")

# Validaciones de seguridad
if "df_resultados_no_supervisados" not in globals():
    raise NameError("df_resultados_no_supervisados no existe. Ejecuta primero la comparación de modelos no supervisados.")

if "df_resultados_supervisados" not in globals():
    raise NameError("df_resultados_supervisados no existe. Ejecuta primero la comparación de modelos supervisados.")

# Crear copias para no alterar las tablas originales
df_no_sup_final = df_resultados_no_supervisados.copy()
df_sup_final = df_resultados_supervisados.copy()

# Agregar tipo de modelo
df_no_sup_final["Tipo Modelo"] = "No supervisado"
df_sup_final["Tipo Modelo"] = "Supervisado"

# Unir resultados
df_resultados_finales = pd.concat(
    [df_no_sup_final, df_sup_final],
    ignore_index=True
)

# Ordenar por Recall, métrica clave en fraude
df_resultados_finales = df_resultados_finales.sort_values(
    by="Recall",
    ascending=False
)

print("\nResultados finales ordenados por Recall:")
display(df_resultados_finales)

# Ordenar por F1 Score
df_final_f1 = df_resultados_finales.sort_values(
    by="F1 Score",
    ascending=False
)

print("\nResultados finales ordenados por F1 Score:")
display(df_final_f1)

# Ordenar por ROC AUC
df_final_auc = df_resultados_finales.sort_values(
    by="ROC AUC",
    ascending=False
)

print("\nResultados finales ordenados por ROC AUC:")
display(df_final_auc)

# Identificar mejores modelos
mejor_final_recall = df_resultados_finales.iloc[0]
mejor_final_f1 = df_final_f1.iloc[0]
mejor_final_auc = df_final_auc.iloc[0]

print("\nMejor modelo general según Recall:")
print(mejor_final_recall["Modelo"], "-", round(mejor_final_recall["Recall"], 4))

print("\nMejor modelo general según F1 Score:")
print(mejor_final_f1["Modelo"], "-", round(mejor_final_f1["F1 Score"], 4))

print("\nMejor modelo general según ROC AUC:")
print(mejor_final_auc["Modelo"], "-", round(mejor_final_auc["ROC AUC"], 4))

print("\nConclusión general:")
print("Los modelos no supervisados ayudan a detectar anomalías y segmentos de riesgo sin usar fraud_label para entrenar.")
print("Los modelos supervisados aprenden directamente de fraud_label para clasificar transacciones.")
print("La selección final del modelo debe considerar Recall, F1 Score, ROC AUC, falsos positivos y falsos negativos.")

COMPARACIÓN FINAL DE TODOS LOS MODELOS

Resultados finales ordenados por Recall:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas,Tipo Modelo
12,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
15,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
18,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
19,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
13,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
20,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
14,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
17,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
16,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
0,Combinado No Supervisado,0.617826,0.153660,0.625960,0.246749,0.621441,31298,172385,18702,203683,No supervisado



Resultados finales ordenados por F1 Score:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas,Tipo Modelo
13,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
19,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
17,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
14,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
20,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
16,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
15,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
12,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
18,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
0,Combinado No Supervisado,0.617826,0.153660,0.625960,0.246749,0.621441,31298,172385,18702,203683,No supervisado



Resultados finales ordenados por ROC AUC:


,Modelo,Accuracy,Precision,Recall,F1 Score,ROC AUC,Fraudes Detectados,Falsos Positivos,Falsos Negativos,Alertas Generadas,Tipo Modelo
12,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
15,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
18,Logistic Regression,0.999847,0.998934,0.999533,0.999234,1.000000,14993,16,7,15009,Supervisado
19,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
13,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
20,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
14,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
17,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
16,Random Forest,0.999900,1.000000,0.999000,0.999500,1.000000,14985,0,15,14985,Supervisado
0,Combinado No Supervisado,0.617826,0.153660,0.625960,0.246749,0.621441,31298,172385,18702,203683,No supervisado



Mejor modelo general según Recall:
Logistic Regression - 0.9995

Mejor modelo general según F1 Score:
Random Forest - 0.9995

Mejor modelo general según ROC AUC:
Logistic Regression - 1.0

Conclusión general:
Los modelos no supervisados ayudan a detectar anomalías y segmentos de riesgo sin usar fraud_label para entrenar.
Los modelos supervisados aprenden directamente de fraud_label para clasificar transacciones.
La selección final del modelo debe considerar Recall, F1 Score, ROC AUC, falsos positivos y falsos negativos.
